In [1]:
import argparse,os,sys
import numpy as np
import neuroboros as nb
from shared_utils.data_io_utils import parse_range,return_Tian_labels,return_aseg_labels,get_dm,get_rois_and_space
from shared_utils.resource_management_utils import print_memory_usage,save_args_to_json,load_args_from_json,merge_args
from shared_utils.hyperalignment_utils import calculate_connectivity,get_target_ts
from hyperalignment import searchlight_template, compute_template

In [2]:
args = load_args_from_json('./config_template_example.json')
print(args)
args = argparse.Namespace(**args)

{'load_json': None, 'dataset': 'Budapest', 'outdir': '.', 'seeds': 'aseg', 'targets': ['aseg', 'onavg-ico8'], 'task': 'budapest', 'run': [1, 2, 3], 'subj': [0, 1], 'template_dir': '', 'searchlight_radius': 20, 'searchlight_center': 'onavg-ico32', 'notweighted': False, 'surface_space': 'onavg-ico32', 'volume_space': 'mni-2mm', 'surface_resample': '1step_pial_overlap', 'volume_resample': '1step_linear_overlap', 'prep': 'default', 'zscore_axis': 'searchlight', 'separate_zscore': True, 'saved_beta_root': '/dartfs/rc/lab/H/HaxbyLab/datasets/', 'dry_run': False, 'stage': 'template', 'saved_beta_path': '/dartfs/rc/lab/H/HaxbyLab/datasets/saved_confound_betas/default/Budapest'}


In [3]:
args.stage = "template"
if 'saved_beta_root' in args and os.path.exists(args.saved_beta_root):
    args.saved_beta_path = os.path.join(args.saved_beta_root,'saved_confound_betas',args.prep,args.dataset)
    print(args.saved_beta_path)

config_path = save_args_to_json(args)

if  args.template_dir:
    template_dir = args.template_dir
else:
    template_dir = os.path.join(args.outdir, "templates")

/dartfs/rc/lab/H/HaxbyLab/datasets/saved_confound_betas/default/Budapest
Configuration saved to: ./config_template_20251103_130527.json


In [4]:
kwargs = { # for older datasets, there might be multiple versions/folders
'space': [args.surface_space, args.volume_space],
'resample': [args.surface_resample,args.volume_resample],
'prep':args.prep
}   
if args.dataset.lower() in ["budapest", "raiders"]:
    dset=nb.datasets.datasets[args.dataset.lower()](fp_version="20.2.7",**kwargs)
elif args.dataset.lower() in nb.datasets.datasets.keys():
    dset=nb.datasets.datasets[args.dataset.lower()](**kwargs)
else:
    ValueError("Unsupported Dataset")
sids = np.array(dset.subjects)[args.subj]

if args.zscore_axis=="searchlight":
    zscore_ax= 0
elif args.zscore_axis=="fullmatrix":
    zscore_ax = None
else:
    raise ValueError("Unsupported zscore_axis")

os.makedirs(args.outdir, exist_ok=True)    
os.makedirs(template_dir, exist_ok=True)

In [5]:
dset.surface_resample

'1step_pial_overlap'

In [ ]:
# Make seed ts
seedrois,space = get_rois_and_space(args.seeds,args.surface_space,args.volume_space)
seed_ts = get_dm(dset, sids, seedrois, args.task, args.run, space,
                atlas=args.seeds,saved_beta_path=args.saved_beta_path)

In [ ]:
all_target_ts = []
for target in args.targets:
    print(target)
    targetrois,space = get_rois_and_space(target,args.surface_space,args.volume_space)
    if target==args.seeds:
        target_ts = seed_ts.copy()
    else:
        target_ts = get_dm(dset, sids, targetrois, args.task, args.run, space,
                atlas=target,saved_beta_path=args.saved_beta_path)
    if target.startswith('onavg'):
        MAPPINGS = {roi:nb.mapping(roi, args.surface_space, target, mask=True) for roi in targetrois}
        target_ts = get_target_ts(target_ts,'mapping',target_item=MAPPINGS)
    else:
        target_ts = get_target_ts(target_ts,'searchlight_mean')                
    all_target_ts.append(target_ts)

In [ ]:
conn = []
for roi in seedrois:
    ts_target = all_target_ts[0][sids[0]]
    ts_seed = seed_ts[sids[0]][roi]
    conn.append(calculate_connectivity(ts_target,ts_seed,zscore_axis=0,metric='correlation'))
conn = np.concatenate(conn,axis=1)
print(conn.shape)

In [ ]:
dm_target = {}
for sid in sids:
    for target in args.targets:
        if target == 'None':
            raise ValueError("Templates not needed for anatomical alignment. Exiting")
        elif target == 'response':
            dm = [seed_ts[sid][seedroi] for seedroi in seedrois]
        else: # target is connectivity
            if target == args.seeds:
                subj_seed_ts = seed_ts[sid]
                # for sid,subj_seed_ts in seed_ts.items():
                subj_target_ts = []
                for targetroi,val in subj_seed_ts.items():
                    subj_target_ts.append(np.mean(subj_seed_ts[targetroi],axis=1))
                subj_target_ts = np.stack(subj_target_ts,axis=1)
                dm = []
                for seedroi in seedrois:
                    if args.separate_zscore:
                        subj_conn = calculate_connectivity(subj_target_ts,subj_seed_ts[seedroi],zscore_axis=zscore_axis_arg,metric ='correlation') # we don't zscore here
                    else:
                        subj_conn = calculate_connectivity(subj_target_ts,subj_seed_ts[seedroi],zscore_axis=None,metric ='correlation') # we zscore later
                    dm.append(subj_conn)        
        dm_target[sid] = dm   

In [ ]:
for sid in sids:
    print(np.concatenate(dm_target[sid],axis=1).shape)

In [ ]:
tmp = [dm_target[sid][0] for sid in sids]
dss = np.stack(tmp)
print(dss.shape)

In [ ]:
ns, nt, nv = dss.shape
X = dss.transpose(1,0,2).reshape(nt,ns*nv)

In [ ]:
X2 = np.concatenate(tmp,axis=1)
print(X2.shape)
np.allclose(X,X2)

In [ ]:
tmp1 = compute_template(dss, kind='pca', max_npc=dss.shape[1], common_topography=False)

In [ ]:
tmp2 = compute_template(tmp, kind='pca', max_npc=dss.shape[1], common_topography=False)

In [ ]:
np.allclose(tmp1,tmp2)

In [ ]:
subj_target_ts = np.stack(subj_target_ts,axis=1)
subj_conn = calculate_connectivity(subj_target_ts,subj_seed_ts[seedroi],zscore_axis=zscore_axis_arg,metric ='correlation') # we don't zscore here

In [ ]:
subj_conn = calculate_connectivity(subj_target_ts[:,np.newaxis],subj_seed_ts[seedroi],zscore_axis=zscore_axis_arg)
print(subj_conn.shape)

In [ ]:
np.nanmean(seed_ts[sids[0]]['l-accumbens'],axis=1)

In [ ]:
print(dm.shape)